# SAM-AI Sovereign Agent: Real Base-Model Smoke Test (Qwen3.8-27B Zero-Shot Base)

**Objective:** Verify that the sovereign SAM-AI agent (`sam_ai.agent`) loads authentic base model weights (`Qwen/Qwen3.8-27B`) via `HuggingFaceModelRunner`, connects to the official ARC-3 `Arcade` environment interface, and executes a strictly budgeted 5-step interactive loop.

> **Hard Rules:**
> 1. **Zero competition submission:** Does not create `submission.parquet` or invoke submission APIs.
> 2. **Base-Model Only:** Does not mount, load, or evaluate the SAM-AI adapter. All results labeled `REAL_MODEL_ZERO_SHOT_BASE`.
> 3. **No mock fallback:** If weights, GPU, or environment data are absent, halts cleanly with `[BLOCKED]`.
> 4. **No silent substitution:** Refuses to evaluate DeepSeek or other checkpoints when Qwen is requested.
> 5. **Small compute footprint:** Bounded to 5 environment steps total (< 2 minutes GPU runtime).
> 6. **Architectural Parity:** Uses identical `HuggingFaceModelRunner` and telemetry schema as the local `scripts/run_real_model_smoke_test.py`.


In [ ]:
"""Find Kaggle inputs by what they contain, not by a guessed path.

Why: the smoke test halted on a path guess ("sam_ai not found", "environment
data not found"). Kaggle mounts differ by runtime (/kaggle/input/<slug> vs
/kaggle/input/datasets/<owner>/<slug> vs /kaggle/input/competitions/<slug> vs
/kaggle/input/models/<owner>/<slug>/<framework>/<variation>/<version>), and a
dataset uploaded as .tar.gz may be mounted already extracted. This module
searches for the thing itself, always prints the layout it saw, and raises with
that layout if anything is missing. It never falls back silently.

Usage (first cell of the notebook, and in scripts/run_real_model_smoke_test.py):

    from kaggle_path_discovery import discover_inputs
    paths = discover_inputs()          # prints tree, raises if anything is missing
    sys.path.insert(0, paths["sam_ai_parent"])
    ENV_DIR   = paths["environment_files"]
    MODEL_DIR = paths["base_model_dir"]

Stdlib only. Does not download anything and does not touch the network.
"""
import json
import os
import sys
import tarfile

INPUT_ROOT = "/kaggle/input"
EXTRACT_DIR = "/kaggle/working/sam_ai_src"
MAX_DEPTH = 8


def _walk(root, max_depth=MAX_DEPTH):
    """os.walk that follows symlinks safely and stops at max_depth."""
    seen = set()
    root = os.path.abspath(root)
    base = root.rstrip(os.sep).count(os.sep)
    for dirpath, dirnames, filenames in os.walk(root, followlinks=True):
        real = os.path.realpath(dirpath)
        if real in seen:
            dirnames[:] = []
            continue
        seen.add(real)
        if dirpath.count(os.sep) - base >= max_depth:
            dirnames[:] = []
        dirnames.sort()
        yield dirpath, dirnames, sorted(filenames)


def tree(root=INPUT_ROOT, max_depth=5, files_per_dir=6, max_lines=300):
    """Human-readable layout, so a failed run still tells us what was mounted."""
    if not os.path.isdir(root):
        return f"{root} does not exist"
    lines, base = [], root.rstrip(os.sep).count(os.sep)
    for dirpath, dirnames, filenames in _walk(root, max_depth):
        depth = dirpath.count(os.sep) - base
        if depth >= max_depth:
            dirnames[:] = []
        lines.append(f"{'  ' * depth}{os.path.basename(dirpath) or dirpath}/")
        shown = filenames[:files_per_dir]
        for f in shown:
            lines.append(f"{'  ' * (depth + 1)}{f}")
        if len(filenames) > len(shown):
            lines.append(f"{'  ' * (depth + 1)}... (+{len(filenames) - len(shown)} files)")
        if len(lines) >= max_lines:
            lines.append("... (tree truncated)")
            break
    return "\n".join(lines)


def _safe_extract(tar_path, dest):
    """Extract a tarball, refusing links, devices and path escapes."""
    os.makedirs(dest, exist_ok=True)
    dest_real = os.path.realpath(dest)
    with tarfile.open(tar_path) as tar:
        members = tar.getmembers()
        for m in members:
            if not (m.isfile() or m.isdir()):
                raise RuntimeError(f"refusing non-regular tar member: {m.name}")
            target = os.path.realpath(os.path.join(dest_real, m.name))
            if target != dest_real and not target.startswith(dest_real + os.sep):
                raise RuntimeError(f"refusing tar member outside destination: {m.name}")
        tar.extractall(dest_real, members=members)
    return dest_real


def find_sam_ai_parent(root=INPUT_ROOT, extract_dir=EXTRACT_DIR):
    """Directory to put on sys.path so `import sam_ai` works.

    Prefers an already-extracted package (Kaggle usually unpacks archives);
    falls back to extracting sam_ai_source_bundle*.tar.gz.
    """
    for dirpath, dirnames, filenames in _walk(root):
        if os.path.basename(dirpath) == "sam_ai" and "__init__.py" in filenames:
            return os.path.dirname(dirpath)
    for dirpath, _, filenames in _walk(root):
        for f in filenames:
            if f.startswith("sam_ai_source_bundle") and f.endswith((".tar.gz", ".tgz")):
                parent = _safe_extract(os.path.join(dirpath, f), extract_dir)
                if os.path.isfile(os.path.join(parent, "sam_ai", "__init__.py")):
                    return parent
    return None


def find_environment_files(root=INPUT_ROOT):
    """A directory named environment_files that really contains game metadata."""
    for dirpath, dirnames, _ in _walk(root):
        if os.path.basename(dirpath) == "environment_files":
            for sub, _, files in _walk(dirpath, max_depth=4):
                if "metadata.json" in files:
                    return dirpath
    return None


class ModelMismatchError(RuntimeError):
    """Raised when discovered model contradicts the explicitly requested target model."""
    pass


def verify_model_identity(model_dir, config=None, expected_model="qwen3.8"):
    """
    Verifies that the discovered model directory and config match the expected model.
    Fails closed if the expected model is missing, or if a different model
    (such as DeepSeek) is discovered when Qwen was expected.
    """
    if not model_dir or not os.path.exists(model_dir):
        raise FileNotFoundError(f"Model directory '{model_dir}' does not exist.")

    if config is None:
        cfg_path = os.path.join(model_dir, "config.json")
        if os.path.exists(cfg_path):
            try:
                with open(cfg_path, encoding="utf-8") as fh:
                    config = json.load(fh)
            except Exception as e:
                raise RuntimeError(f"Corrupt or unreadable config.json in '{model_dir}': {e}")
        else:
            config = {}

    expected_lower = (expected_model or "").lower()
    dir_lower = str(model_dir).lower()
    name_or_path = str(config.get("_name_or_path", "")).lower()
    model_type = str(config.get("model_type", "")).lower()

    if any(k in expected_lower for k in ("qwen3.8", "qwen38", "qwen3", "qwen")):
        # Refuse DeepSeek silent substitution
        if "deepseek" in dir_lower or "deepseek" in name_or_path:
            raise ModelMismatchError(
                f"MODEL IDENTITY MISMATCH: Smoke test is explicitly configured for Qwen3.8 ('{expected_model}'), "
                f"but discovered DeepSeek checkpoint in '{model_dir}'. "
                f"Refusing silent model substitution. To evaluate DeepSeek, set target_model='deepseek' explicitly."
            )
        is_qwen = ("qwen" in dir_lower or "qwen" in model_type or "qwen" in name_or_path)
        if not is_qwen:
            raise ModelMismatchError(
                f"MODEL IDENTITY MISMATCH: Expected Qwen model for target '{expected_model}', "
                f"but discovered incompatible model in '{model_dir}' (model_type='{model_type}')."
            )
        return {
            "verified": True,
            "expected_model": expected_model,
            "discovered_dir": model_dir,
            "model_type": model_type,
            "family": "qwen3.8" if ("qwen3.8" in dir_lower or "qwen3.8" in name_or_path) else "qwen",
        }
    elif any(k in expected_lower for k in ("deepseek", "unsloth-deepseek")):
        is_deepseek = ("deepseek" in dir_lower or "deepseek" in name_or_path)
        if not is_deepseek:
            raise ModelMismatchError(
                f"MODEL IDENTITY MISMATCH: Explicitly configured for DeepSeek baseline ('{expected_model}'), "
                f"but discovered non-DeepSeek model in '{model_dir}'."
            )
        return {
            "verified": True,
            "expected_model": expected_model,
            "discovered_dir": model_dir,
            "model_type": model_type,
            "family": "deepseek_r1_distill",
        }
    return {
        "verified": True,
        "expected_model": expected_model,
        "discovered_dir": model_dir,
        "model_type": model_type,
        "family": "generic",
    }


def find_base_model(root=INPUT_ROOT, name_hint=None, target_model=None):
    """A directory with config.json + safetensors whose path mentions the model.

    Returns (dir, summary) where summary records what the config says, so the
    run log shows exactly which weights were found. Never picks a directory
    that does not match the candidate hints (no silent substitution with wrong models).
    Prioritizes primary Qwen3.8-27B, then baseline DeepSeek-14B when unspecified.
    When target_model is specified, searches strictly for that model family.
    """
    if target_model:
        t_lower = target_model.lower()
        if any(k in t_lower for k in ("qwen3.8", "qwen38", "qwen3")):
            hints = ["qwen3.8-27b", "qwen3.8", "qwen3"]
        elif "deepseek" in t_lower:
            hints = ["deepseek-r1-distill-qwen-14b", "unsloth-deepseek", "deepseek"]
        else:
            hints = [target_model]
    elif name_hint:
        hints = [name_hint]
    else:
        # Priority order when unspecified: primary research checkpoint first, then legacy baseline
        hints = ["qwen3.8-27b", "qwen3.8", "deepseek-r1-distill-qwen-14b", "unsloth-deepseek"]

    for hint in hints:
        for dirpath, _, filenames in _walk(root):
            if "config.json" in filenames and any(f.endswith(".safetensors") for f in filenames):
                if hint.lower() not in dirpath.lower():
                    continue
                try:
                    with open(os.path.join(dirpath, "config.json"), encoding="utf-8") as fh:
                        cfg = json.load(fh)
                except (OSError, ValueError):
                    continue
                q = cfg.get("quantization_config") or {}
                return dirpath, {
                    "matched_hint": hint,
                    "model_type": cfg.get("model_type"),
                    "hidden_size": cfg.get("hidden_size"),
                    "num_hidden_layers": cfg.get("num_hidden_layers"),
                    "quant_method": q.get("quant_method"),
                    "load_in_4bit": q.get("load_in_4bit"),
                    "bnb_4bit_quant_type": q.get("bnb_4bit_quant_type"),
                }
    return None, None


def discover_inputs(root=INPUT_ROOT, verbose=True, name_hint=None, target_model=None):
    layout = tree(root)
    if verbose:
        print(f"--- layout of {root} ---\n{layout}\n--- end layout ---")
    sam_parent = find_sam_ai_parent(root)
    env_dir = find_environment_files(root)
    model_dir, model_cfg = find_base_model(root, name_hint=name_hint, target_model=target_model)
    found = {
        "sam_ai_parent": sam_parent,
        "environment_files": env_dir,
        "base_model_dir": model_dir,
        "base_model_config": model_cfg,
    }
    missing = [k for k in ("sam_ai_parent", "environment_files", "base_model_dir") if not found[k]]
    if missing:
        target_info = f" for target '{target_model or name_hint}'" if (target_model or name_hint) else ""
        raise RuntimeError(
            f"Missing inputs: {missing}{target_info}. Nothing was substituted. Layout seen:\n{layout}"
        )
    if verbose:
        print(json.dumps(found, indent=2))
    return found


if __name__ == "__main__":
    try:
        discover_inputs()
    except RuntimeError as e:
        print(e, file=sys.stderr)
        sys.exit(1)


In [ ]:
# ==============================================================================
# 1. Environment, Dependencies & Static Precondition Audit
# ==============================================================================
import os
import sys
import time
import json
import subprocess
from pathlib import Path

print('=' * 75)
print('  SAM-AI REAL BASE-MODEL SMOKE TEST: STATIC PRECONDITION AUDIT')
print('=' * 75)

# 0. Explicit Model Selection & Content Discovery
TARGET_MODEL = 'qwen3.8-27b'  # Explicit primary checkpoint. Baseline requires explicit change to 'deepseek_r1_distill_14b'
paths = discover_inputs(target_model=TARGET_MODEL, name_hint=TARGET_MODEL)
sam_ai_parent = str(paths['sam_ai_parent'])
if sam_ai_parent not in sys.path:
    sys.path.insert(0, sam_ai_parent)

base_model_path = str(paths['base_model_dir'])
ENV_DATA_DIR = str(paths['environment_files'])

# Strict Model Identity Verification: reject silent DeepSeek substitution
model_verif = verify_model_identity(base_model_path, paths.get('base_model_config'), expected_model=TARGET_MODEL)
print(f'[OK] Target Model Configured: {TARGET_MODEL}')
print(f'[OK] Model Identity Verified: {model_verif["family"]} ({base_model_path})')
print(f'[OK] sam_ai parent resolved:   {sam_ai_parent}')
print(f'[OK] Base Model resolved:      {base_model_path}')
print(f'[OK] Environment dir resolved: {ENV_DATA_DIR}')

# 1. Check and Install Official arc-agi / arcengine Wheels Offline
try:
    import arc_agi
    import arcengine
    print('[OK] arc-agi and arcengine packages are available.')
except ImportError:
    wheel_dirs = [
        Path(ENV_DATA_DIR).parent / 'arc_agi_3_wheels',
    ]
    installed = False
    for wd in wheel_dirs:
        if wd.exists() and list(wd.glob('*.whl')):
            print(f'[*] Installing offline wheels from {wd}...')
            res = subprocess.run([
                sys.executable, '-m', 'pip', 'install', '--no-index',
                f'--find-links={wd}', 'arc-agi', 'arcengine'
            ], capture_output=True, text=True)
            if res.returncode == 0:
                print('[OK] arc-agi and arcengine installed from offline wheels.')
                installed = True
                break
            else:
                print(f'[WARN] Offline wheel installation warning: {res.stderr}')
    if not installed:
        raise RuntimeError('arc-agi and arcengine packages not found and offline wheels missing.')

# 2. Check GPU Accelerator & Memory
try:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is not available. GPU accelerator must be enabled in Notebook Settings.')
    else:
        free_b, total_b = torch.cuda.mem_get_info()
        gpu_name = torch.cuda.get_device_name(0)
        print(f'[OK] GPU Accelerator: {gpu_name}')
        print(f'[OK] Total VRAM:      {total_b / (1024**3):.2f} GB (Free: {free_b / (1024**3):.2f} GB)')
        print(f'[OK] bfloat16:        {"Supported" if torch.cuda.is_bf16_supported() else "Not Supported"}')
except ImportError:
    raise RuntimeError('PyTorch is not installed.')

# 3. Adapter Explicitly Disabled (Zero-Shot Base Mode)
adapter_path = None
adapter_bytes = 0
print('[INFO] Operating in Base-Model-Only Zero-Shot Mode (REAL_MODEL_ZERO_SHOT_BASE).')
print('       SAM-AI LoRA adapter is NOT loaded in this evaluation.')

# 4. Check Official ARC-3 Environment Data via Arcade
found_envs = len(list(Path(ENV_DATA_DIR).rglob('metadata.json')))
print(f'[OK] Found {found_envs} official ARC-3 environment files in {ENV_DATA_DIR}.')
print('\n[READY] All hardware, base model, and environment prerequisites verified.')


In [ ]:
# ==============================================================================
# 2. Sovereign ModelRunner Loading (Base Model Only)
# ==============================================================================
from sam_ai.substrate.models import HuggingFaceModelRunner

print(f'[*] Initializing sovereign HuggingFaceModelRunner with base model...')
runner = HuggingFaceModelRunner(
    model_name_or_path=base_model_path,
    adapter_path=None, # Strictly base-model zero-shot
    device_map='auto',
    load_in_4bit=True,
)
runner.load_model()

is_adapter_active = runner.is_adapter_loaded
assert not is_adapter_active, 'CRITICAL: Adapter must not be loaded in zero-shot base evaluation.'
print(f'[OK] ModelRunner ready.')
print(f'     Model Name:     {runner.model_name}')
print(f'     Adapter Active: {is_adapter_active} (Zero-Shot Base Mode)')


In [ ]:
eval_type = 'REAL_MODEL_ZERO_SHOT_BASE'
start_kernel_time = time.perf_counter()
WALL_CLOCK_LIMIT_SECONDS = 1500.0  # 25 minutes
results = {}
out_dir = Path('evidence/real_model_smoke_test')
out_dir.mkdir(parents=True, exist_ok=True)

# ==============================================================================
# 3. Official Arcade Setup
# ==============================================================================
from arc_agi import Arcade, OperationMode

arcade = Arcade(
    operation_mode=OperationMode.OFFLINE,
    environments_dir=ENV_DATA_DIR,
)
available_envs = arcade.get_environments()
if not available_envs:
    raise RuntimeError(f'No environments found in {ENV_DATA_DIR}')

chosen_game = available_envs[0].game_id
print(f'[*] Selected smoke-test game: {chosen_game}')

# ==============================================================================
# 4. Arm A: Plain Decoding (Max 5 Steps)
# ==============================================================================
from sam_ai.agent.arc3_agent import SAMARC3Agent

print('=' * 75)
print('  STARTING ARM A: PLAIN DECODING (decode_mode="plain")')
print('=' * 75)
agent_a = SAMARC3Agent(
    model_runner=runner,
    enable_verification_guard=True,
    enable_failure_memory=False,
    is_real_model=True,
    temperature=0.6,
    top_p=0.95,
    seed=42,
    max_tokens=3072,
    use_system_prompt=False,
    stop_on_parse_failure=True,
    decode_mode='plain',
)
env_a = arcade.make(game_id=chosen_game, seed=0)
if env_a is None:
    raise RuntimeError(f'arcade.make returned None for game {chosen_game} (Arm A)')

t0_a = time.perf_counter()
summary_a = agent_a.run_session(
    env_a, game_id=chosen_game, max_steps=5,
    wall_clock_limit_seconds=WALL_CLOCK_LIMIT_SECONDS, kernel_start_time=start_kernel_time
)
elapsed_a = time.perf_counter() - t0_a

report_a = {
    'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'arm': 'Arm A (plain)',
    'decode_mode': 'plain',
    'game_id': chosen_game,
    'evaluation_type': eval_type,
    'is_real_model': True,
    'adapter_loaded': False,
    'adapter_path': None,
    'adapter_bytes': 0,
    'base_model': base_model_path,
    'steps_taken': summary_a.total_steps,
    'levels_completed': summary_a.levels_completed,
    'final_state': summary_a.final_state,
    'success': summary_a.success,
    'verification_failures': summary_a.verification_failures,
    'elapsed_seconds': elapsed_a,
    'action_history': [
        {'step': s.step_idx, 'action': s.executed_action, 'status': s.verification_status.value}
        for s in summary_a.steps_history
    ],
    'steps_telemetry': [
        step.telemetry.to_dict() for step in summary_a.steps_history if step.telemetry is not None
    ],
}

with open(out_dir / 'smoke_summary_arm_a.json', 'w', encoding='utf-8') as f:
    json.dump(report_a, f, indent=2)
with open('smoke_summary_arm_a.json', 'w', encoding='utf-8') as f:
    json.dump(report_a, f, indent=2)
with open(out_dir / 'smoke_trace_arm_a.jsonl', 'a', encoding='utf-8') as f:
    f.write(json.dumps(report_a) + '\n')
with open('smoke_trace_arm_a.jsonl', 'a', encoding='utf-8') as f:
    f.write(json.dumps(report_a) + '\n')

results['arm_a'] = report_a
print(f'[OK] Arm A Complete: Steps={summary_a.total_steps} | State={summary_a.final_state}')

# ==============================================================================
# 5. Arm B: Think-Then-Commit Decoding (Max 5 Steps, Fresh Environment)
# ==============================================================================
time_spent = time.perf_counter() - start_kernel_time
if time_spent >= WALL_CLOCK_LIMIT_SECONDS:
    print(f'[WALL_CLOCK_LIMIT] Elapsed {time_spent:.2f}s >= {WALL_CLOCK_LIMIT_SECONDS}s. Skipping Arm B.')
    report_b = {
        'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        'arm': 'Arm B (think_commit)',
        'decode_mode': 'think_commit',
        'game_id': chosen_game,
        'evaluation_type': eval_type,
        'final_state': 'WALL_CLOCK_LIMIT',
        'skipped': True,
        'reason': 'WALL_CLOCK_LIMIT',
    }
    results['arm_b'] = report_b
    with open(out_dir / 'smoke_summary_arm_b.json', 'w', encoding='utf-8') as f:
        json.dump(report_b, f, indent=2)
    with open('smoke_summary_arm_b.json', 'w', encoding='utf-8') as f:
        json.dump(report_b, f, indent=2)
else:
    print('=' * 75)
    print('  STARTING ARM B: THINK-THEN-COMMIT DECODING (decode_mode="think_commit")')
    print('=' * 75)
    agent_b = SAMARC3Agent(
        model_runner=runner,
        enable_verification_guard=True,
        enable_failure_memory=False,
        is_real_model=True,
        temperature=0.6,
        top_p=0.95,
        seed=42,
        max_tokens=3072,
        use_system_prompt=False,
        stop_on_parse_failure=True,
        decode_mode='think_commit',
    )
    env_b = arcade.make(game_id=chosen_game, seed=0)
    if env_b is None:
        raise RuntimeError(f'arcade.make returned None for game {chosen_game} (Arm B)')

    t0_b = time.perf_counter()
    summary_b = agent_b.run_session(
        env_b, game_id=chosen_game, max_steps=5,
        wall_clock_limit_seconds=WALL_CLOCK_LIMIT_SECONDS, kernel_start_time=start_kernel_time
    )
    elapsed_b = time.perf_counter() - t0_b

    report_b = {
        'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        'arm': 'Arm B (think_commit)',
        'decode_mode': 'think_commit',
        'game_id': chosen_game,
        'evaluation_type': eval_type,
        'is_real_model': True,
        'adapter_loaded': False,
        'adapter_path': None,
        'adapter_bytes': 0,
        'base_model': base_model_path,
        'steps_taken': summary_b.total_steps,
        'levels_completed': summary_b.levels_completed,
        'final_state': summary_b.final_state,
        'success': summary_b.success,
        'verification_failures': summary_b.verification_failures,
        'elapsed_seconds': elapsed_b,
        'action_history': [
            {'step': s.step_idx, 'action': s.executed_action, 'status': s.verification_status.value}
            for s in summary_b.steps_history
        ],
        'steps_telemetry': [
            step.telemetry.to_dict() for step in summary_b.steps_history if step.telemetry is not None
        ],
    }

    with open(out_dir / 'smoke_summary_arm_b.json', 'w', encoding='utf-8') as f:
        json.dump(report_b, f, indent=2)
    with open('smoke_summary_arm_b.json', 'w', encoding='utf-8') as f:
        json.dump(report_b, f, indent=2)
    with open(out_dir / 'smoke_trace_arm_b.jsonl', 'a', encoding='utf-8') as f:
        f.write(json.dumps(report_b) + '\n')
    with open('smoke_trace_arm_b.jsonl', 'a', encoding='utf-8') as f:
        f.write(json.dumps(report_b) + '\n')

    results['arm_b'] = report_b
    print(f'[OK] Arm B Complete: Steps={summary_b.total_steps} | State={summary_b.final_state}')

# ==============================================================================
# 6. Combined Summaries
# ==============================================================================
with open(out_dir / 'smoke_summary.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)
with open('smoke_summary.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)

with open(out_dir / 'smoke_trace.jsonl', 'a', encoding='utf-8') as f:
    f.write(json.dumps(results) + '\n')
with open('smoke_trace.jsonl', 'a', encoding='utf-8') as f:
    f.write(json.dumps(results) + '\n')

print('=' * 75)
print('TWO-ARM SMOKE TEST EXECUTION FINISHED.')
print(f'Summary file: {out_dir / "smoke_summary.json"}')
print('=' * 75)
